In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import os

#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

In [ ]:
!pip install lpips pytorch-fid --quiet

In [ ]:
import numpy as np
import torch

print("NumPy version :", np.__version__)
print("PyTorch version:", torch.__version__)

In [ ]:
# ── Cell 5 ──────────────────────────────────────────────
!git clone https://github.com/researchmm/AOT-GAN-for-Inpainting.git
%cd AOT-GAN-for-Inpainting

In [ ]:
print(os.listdir("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/"))

In [ ]:

train_dir = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
val_dir   = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017"

In [ ]:

# Dataset class with irregular brush stroke masks
import os
import random
import numpy as np
import cv2
from PIL import Image
import torch
from torch.utils.data import Dataset

class CocoInpaintingDataset(Dataset):
    def __init__(self, image_dir, num_images=None, img_size=(256,256), transform=None):
        self.image_dir = image_dir
        self.images    = sorted(os.listdir(image_dir))
        if num_images:
            self.images = self.images[:num_images]
        self.img_size  = img_size
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path   = os.path.join(self.image_dir, self.images[idx])
        img        = Image.open(img_path).convert("RGB")
        img        = img.resize(self.img_size)
        img_tensor = self.transform(img)

        mask        = self.generate_random_mask(self.img_size)
        mask_tensor = torch.from_numpy(mask).float().unsqueeze(0)

        return img_tensor, mask_tensor

    def generate_random_mask(self, size):
        h, w  = size
        mask  = np.zeros((h, w), np.uint8)

        num_strokes = random.randint(3, 6)
        for _ in range(num_strokes):
            x     = random.randint(0, w)
            y     = random.randint(0, h)
            length  = random.randint(20, 80)
            brush_w = random.randint(10, 30)
            angle   = random.uniform(0, 2 * np.pi)

            for _ in range(length):
                x      = int(x + np.cos(angle) * 4)
                y      = int(y + np.sin(angle) * 4)
                angle += random.uniform(-0.4, 0.4)
                x = int(np.clip(x, 0, w - 1))
                y = int(np.clip(y, 0, h - 1))
                cv2.circle(mask, (x, y), brush_w, 1, -1)

        return mask

In [ ]:

IMG_SIZE = (256, 256)

def pil_to_tensor(img):
    arr = np.array(img).astype(np.float32) / 255.0
    arr = np.ascontiguousarray(arr)
    return torch.from_numpy(arr).permute(2, 0, 1)

transform = pil_to_tensor

train_dataset = CocoInpaintingDataset(
    train_dir, num_images=5000, img_size=IMG_SIZE, transform=transform)

val_dataset = CocoInpaintingDataset(
    val_dir, num_images=500, img_size=IMG_SIZE, transform=transform)

In [ ]:

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2)

In [ ]:

x, m = train_dataset[0]
print("Image tensor shape:", x.shape, x.dtype)
print("Mask tensor shape :", m.shape, m.dtype)


In [ ]:

# Load AOT-GAN model from cloned repo
import importlib.util
import sys

sys.path.append("/kaggle/working/AOT-GAN-for-Inpainting/src/model")

repo_model_dir = "/kaggle/working/AOT-GAN-for-Inpainting/src/model"
common_path    = os.path.join(repo_model_dir, "common.py")
aotgan_path    = os.path.join(repo_model_dir, "aotgan.py")

spec_common = importlib.util.spec_from_file_location("common", common_path)
common      = importlib.util.module_from_spec(spec_common)
spec_common.loader.exec_module(common)
sys.modules["common"] = common

with open(aotgan_path, "r") as f:
    code = f.read()

code = code.replace("from .common import BaseNetwork", "from common import BaseNetwork")

spec_aot = importlib.util.spec_from_loader("aotgan", loader=None)
aotgan   = importlib.util.module_from_spec(spec_aot)
exec(code, aotgan.__dict__)
sys.modules["aotgan"] = aotgan

InpaintGenerator = aotgan.InpaintGenerator
Discriminator    = aotgan.Discriminator

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class Args:
    block_num = 8
    rates     = [1, 2, 4, 8]

args      = Args()
generator = InpaintGenerator(args).to(device)


pretrained_weights = "/kaggle/input/datasets/sarikajs/pretrained-wg1/G0000000.pt"
D_pretrained       = "/kaggle/input/datasets/sarikajs/pretrained-wg1/D0000000.pt"

generator.load_state_dict(torch.load(pretrained_weights, map_location=device))
print("Pretrained generator weights loaded!")

criterion = nn.L1Loss()
optimizer = optim.Adam(generator.parameters(), lr=2e-4)

In [ ]:

# Generator-only training (warm-up phase)
num_epochs = 10

for epoch in range(num_epochs):
    generator.train()
    for i, (imgs, masks) in enumerate(train_loader):
        imgs  = imgs.to(device)
        masks = masks.to(device)

        masked_imgs = imgs * (1 - masks)         # corrupt input
        outputs     = generator(masked_imgs, masks)  #  generator sees masked image
        loss        = criterion(outputs, imgs)    # full image reconstruction

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Step [{i}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")

    # Validation
    generator.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            masked_imgs = imgs * (1 - masks)         #  corrupt input
            outputs     = generator(masked_imgs, masks)  #  masked input
            val_loss   += criterion(outputs, imgs).item()  #  full loss

        val_loss /= len(val_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Validation Loss: {val_loss:.4f}")

torch.save(generator.state_dict(), "/kaggle/working/aotgan_finetuned_coco.pt")
print("Generator saved!")

In [ ]:

# Load discriminator with pretrained weights
discriminator = Discriminator().to(device)
discriminator.load_state_dict(
    torch.load("/kaggle/input/datasets/sarikajs/pretrained-wg1/D0000000.pt", map_location=device))
print("Pretrained discriminator weights loaded!")

In [ ]:

g_optim = optim.Adam(generator.parameters(),     lr=2e-4, betas=(0.5, 0.999))
d_optim = optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.999))

In [ ]:

def gan_hinge_loss_dis(real_pred, fake_pred):
    loss_real = torch.mean(nn.ReLU()(1.0 - real_pred))
    loss_fake = torch.mean(nn.ReLU()(1.0 + fake_pred))
    return loss_real + loss_fake

def gan_hinge_loss_gen(fake_pred):
    return -torch.mean(fake_pred)

In [ ]:

# Full GAN training (generator + discriminator)
num_epochs = 5

for epoch in range(num_epochs):
    generator.train()
    discriminator.train()

    for i, (imgs, masks) in enumerate(train_loader):
        imgs  = imgs.to(device)
        masks = masks.to(device)

        masked_imgs = imgs * (1 - masks)             #  corrupt input once, reuse

        # ── Train Discriminator ──────────────────
        d_optim.zero_grad()

        with torch.no_grad():
            fake_imgs = generator(masked_imgs, masks)    #  masked input

        real_pred = discriminator(imgs)
        fake_pred = discriminator(fake_imgs.detach())    #  detach

        d_loss = gan_hinge_loss_dis(real_pred, fake_pred)
        d_loss.backward()
        d_optim.step()

        # ── Train Generator ──────────────────────
        g_optim.zero_grad()

        fake_imgs = generator(masked_imgs, masks)        #  masked input
        fake_pred = discriminator(fake_imgs)

        adv_loss = gan_hinge_loss_gen(fake_pred)
        l1_loss  = criterion(fake_imgs, imgs)            #  full image loss
        g_loss   = adv_loss + 10 * l1_loss

        g_loss.backward()
        g_optim.step()

        if i % 50 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Step {i}/{len(train_loader)} | "
                  f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f} | "
                  f"ADV: {adv_loss.item():.4f} | L1: {l1_loss.item():.4f}")
    # Validation
    generator.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)

            masked_imgs = imgs * (1 - masks)             #  corrupt input
            outputs     = generator(masked_imgs, masks)  #  masked input
            val_loss   += criterion(outputs, imgs).item()  #  full loss

        val_loss /= len(val_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Validation L1 Loss: {val_loss:.4f}")

torch.save(generator.state_dict(), "/kaggle/working/aotgan_G_finetuned.pt")
torch.save(discriminator.state_dict(), "/kaggle/working/aotgan_D_finetuned.pt")
print("Generator + Discriminator saved!")

In [ ]:

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = InpaintGenerator(args).to(device)
generator.load_state_dict(
    torch.load("/kaggle/working/aotgan_G_finetuned.pt", map_location=device))
generator.eval()

def to_np(t):
    return np.clip(t.detach().cpu().numpy().transpose(1, 2, 0), 0, 1)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for row in range(3):
    sample_img, sample_mask = val_dataset[row]

    input_img  = sample_img.unsqueeze(0).to(device)
    input_mask = sample_mask.unsqueeze(0).to(device)

    masked_input = input_img * (1 - input_mask)      #  corrupt before inference

    with torch.no_grad():
        output = generator(masked_input, input_mask)  #  masked input

    axes[row][0].imshow(to_np(sample_img));       axes[row][0].set_title("Original");         axes[row][0].axis("off")
    axes[row][1].imshow(to_np(masked_input[0]));  axes[row][1].set_title("Masked Input");     axes[row][1].axis("off")
    axes[row][2].imshow(to_np(output[0]));        axes[row][2].set_title("Inpainted Output"); axes[row][2].axis("off")

plt.suptitle("AOT-GAN Inpainting Results — COCO 2017", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("inpainting_results.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

!pip install lpips pytorch-fid --quiet

In [ ]:

import torch
import lpips
import numpy as np
from math import log10
from skimage.metrics import structural_similarity as ssim

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lpips_loss = lpips.LPIPS(net='alex').to(device)

def calc_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2).item()
    if mse == 0:
        return 100.0
    return 20 * log10(1.0 / np.sqrt(mse))

def calc_ssim(img1, img2):
    i1 = img1.permute(1, 2, 0).cpu().numpy()
    i2 = img2.permute(1, 2, 0).cpu().numpy()
    return ssim(i1, i2, channel_axis=-1, data_range=1.0)

def calc_lpips(img1, img2):
    return lpips_loss(img1.unsqueeze(0), img2.unsqueeze(0)).item()

In [ ]:

# Final evaluation on validation set
psnr_list, ssim_list, lpips_list = [], [], []

generator.eval()

for idx, (gt, mask) in enumerate(val_dataset):
    gt   = gt.to(device).unsqueeze(0)
    mask = mask.to(device).unsqueeze(0)

    masked_input = gt * (1 - mask)              #  corrupt before inference

    with torch.no_grad():
        pred = generator(masked_input, mask)    #  masked input

    gt_img   = gt[0]
    pred_img = pred[0]

    psnr_list.append(calc_psnr(gt_img, pred_img))
    ssim_list.append(calc_ssim(gt_img, pred_img))
    lpips_list.append(calc_lpips(gt_img, pred_img))

print("=== AOT-GAN Evaluation Results (COCO 2017 val) ===")
print(f"PSNR  : {np.mean(psnr_list):.4f} ± {np.std(psnr_list):.4f} dB")
print(f"SSIM  : {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")
print(f"LPIPS : {np.mean(lpips_list):.4f} ± {np.std(lpips_list):.4f}")

In [ ]:
!pip install gradio --quiet

import gradio as gr
import torch
import numpy as np
from PIL import Image

def inpaint_image(data):
    try:
        # Extract image and user-drawn mask
        image = data["background"]
        layers = data.get("layers", [])

        img = Image.fromarray(image).resize((256, 256)).convert("RGB")
        img_tensor = pil_to_tensor(img).unsqueeze(0).to(device)

        # Build mask from drawn layer or fallback to center box
        if layers and layers[0] is not None:
            mask_raw = layers[0]
            if mask_raw.shape[2] == 4:
                alpha = mask_raw[:, :, 3]
            else:
                alpha = mask_raw[:, :, 0]
            mask_np = Image.fromarray(alpha).resize((256, 256))
            mask_np = np.array(mask_np)
            mask_tensor = torch.tensor((mask_np > 10).astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
        else:
            # Fallback: center rectangle
            mask_tensor = torch.zeros(1, 1, 256, 256).to(device)
            mask_tensor[:, :, 80:180, 80:180] = 1

        # Masked input
        masked_input = img_tensor * (1 - mask_tensor)

        # Inpaint
        with torch.no_grad():
            output = generator(masked_input, mask_tensor)

        # Convert tensors to numpy images
        def tensor_to_img(t):
            arr = t[0].permute(1, 2, 0).cpu().numpy()
            return (np.clip(arr, 0, 1) * 255).astype(np.uint8)

        original_img    = np.array(img)
        masked_img      = tensor_to_img(masked_input)
        inpainted_img   = tensor_to_img(output)

        # Side-by-side: Original | Masked | Inpainted
        combined = np.concatenate([original_img, masked_img, inpainted_img], axis=1)
        return combined

    except Exception as e:
        print(f"Error: {e}")
        blank = np.zeros((256, 768, 3), dtype=np.uint8)
        return blank


gr.Interface(
    fn=inpaint_image,
    inputs=gr.ImageEditor(
        type="numpy",
        label="Upload image & draw over region to inpaint",
        brush=gr.Brush(colors=["#FFFFFF"], color_mode="fixed", default_size=20)
    ),
    outputs=gr.Image(
        type="numpy",
        label="Original  |  Masked  |  Inpainted"
    ),
    title="🖼️ Image Inpainting using AOT-GAN",
    description=(
        "Upload an image and **paint over** the region you want removed. "
        "AOT-GAN (fine-tuned on COCO 2017) will intelligently fill it in. "
        "Output shows: Original | Masked | Inpainted side by side."
    ),
    examples=None,
    flagging_mode="never"
).launch(share=True)